# SAR-to-EO Image Translation — GalaxEye Assignment
**Model:** Pix2Pix U-Net + PatchGAN  
**Loss:** L1 + Adversarial + FFT Frequency + VGG Perceptual  
**Dataset:** Sentinel-1&2 terrain-split (Kaggle)


In [ ]:
# ================================================================
# CELL 1: Check GPU
# ================================================================
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ================================================================
# CELL 2: Install packages
# Note: dependency conflict warnings are NORMAL — not errors
# ================================================================
!pip install lpips pytorch-fid scikit-image -q

In [ ]:
# ================================================================
# CELL 3: Fresh clone (removes old copy to avoid git conflicts)
# ================================================================
import os, shutil, subprocess

REPO_DIR = '/kaggle/working/sar2eo'
REPO_URL = 'https://github.com/Trafalgar-2006/sar2eo.git'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('Removed old repo.')

subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
os.makedirs('outputs',     exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('logs',        exist_ok=True)
print('Working dir:', os.getcwd())

In [ ]:
# ================================================================
# CELL 3b: Resume from previous session
#
# FRESH START (first ever run): This cell safely does nothing.
#
# RESUMING after session timeout:
#   1. Go to your PREVIOUS notebook on Kaggle
#   2. Right panel → Output tab → click '...' → 'Use as Input'
#      (it mounts the previous output under /kaggle/input/)
#   3. Run this cell — it auto-copies all checkpoints
#   4. Skip Cell 5 (smoke test) and run only the cells that didn't finish
#      The training script auto-resumes from the latest saved epoch.
# ================================================================
import os, shutil, glob

# Find .pth files from any previous notebook output mounted as input
prev_ckpts = glob.glob('/kaggle/input/*/checkpoints/**/*.pth', recursive=True)

if not prev_ckpts:
    print('No previous checkpoints found — fresh start, nothing to restore.')
else:
    print(f'Found {len(prev_ckpts)} checkpoint(s) from previous session. Restoring...')
    for src in prev_ckpts:
        # Keep path from 'checkpoints/' onward
        rel = src[src.index('checkpoints/'):]
        dst = os.path.join('/kaggle/working/sar2eo', rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'  Restored: {rel}')
    print('\nDone! Training cells will auto-resume from the latest saved epoch.')

In [ ]:
# ================================================================
# CELL 4: Set config (dataset path + training settings)
# ================================================================
import yaml, os

# The Sentinel-1&2 dataset mounts here on Kaggle
DATASET_PATH = '/kaggle/input/datasets/requiemonk/sentinel12-image-pairs-segregated-by-terrain/v_2'

if not os.path.exists(DATASET_PATH):
    # Auto-detect if path changed
    for root, dirs, _ in os.walk('/kaggle/input'):
        if any(d in ['agri', 'barrenland', 'grassland', 'urban'] for d in dirs):
            DATASET_PATH = root
            print(f'Auto-detected: {DATASET_PATH}')
            break

assert os.path.exists(DATASET_PATH), f'Dataset not found at {DATASET_PATH}. Add it via + Add Input.'

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['dataset_type']   = 'kaggle'
cfg['data']['kaggle_root']    = DATASET_PATH
cfg['data']['train_terrain']  = ['agri', 'barrenland', 'grassland']
cfg['data']['val_terrain']    = ['urban']
cfg['data']['test_terrain']   = ['urban']
cfg['data']['subset_size']    = None
cfg['training']['batch_size'] = 4
cfg['training']['epochs']     = 100
cfg['training']['num_workers'] = 2

with open('config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config saved!')
print(f"  kaggle_root  : {cfg['data']['kaggle_root']}")
print(f"  train_terrain: {cfg['data']['train_terrain']}")
print(f"  batch_size   : {cfg['training']['batch_size']}")
print(f"  epochs       : {cfg['training']['epochs']}")

In [ ]:
# ================================================================
# CELL 5: Smoke test — 2 epochs, 100 pairs
# SKIP this cell when resuming a previous session.
# Only run on a fresh start to verify the pipeline works.
# ================================================================
import yaml, subprocess, copy

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg_s = copy.deepcopy(cfg)
cfg_s['data']['subset_size']   = 100
cfg_s['training']['epochs']    = 2
cfg_s['training']['val_freq']  = 1
cfg_s['training']['save_freq'] = 2
cfg_s['active_ablation']       = 'full'

with open('config_smoke.yaml', 'w') as f:
    yaml.dump(cfg_s, f)

print('Running smoke test (2 epochs, 100 pairs)...')
r = subprocess.run(['python', 'train.py', '--config', 'config_smoke.yaml', '--ablation', 'full'])

if r.returncode != 0:
    raise RuntimeError('Smoke test FAILED — fix the error above before training.')
print('\nSmoke test PASSED!')

In [ ]:
# ================================================================
# CELL 6a: Config A — L1 only (baseline)
# Auto-resumes from latest checkpoint if already partially trained.
# ================================================================
import subprocess
print('Training Config A: L1 only...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_only'])
print('Config A', 'DONE' if r.returncode == 0 else 'FAILED')

In [ ]:
# ================================================================
# CELL 6b: Config B — L1 + Adversarial
# Auto-resumes from latest checkpoint if already partially trained.
# ================================================================
import subprocess
print('Training Config B: L1 + Adversarial...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv'])
print('Config B', 'DONE' if r.returncode == 0 else 'FAILED')

In [ ]:
# ================================================================
# CELL 6c: Config C — L1 + Adversarial + FFT
# Auto-resumes from latest checkpoint if already partially trained.
# ================================================================
import subprocess
print('Training Config C: L1 + Adversarial + FFT...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'l1_adv_fft'])
print('Config C', 'DONE' if r.returncode == 0 else 'FAILED')

In [ ]:
# ================================================================
# CELL 6d: Config D — Full model (MAIN SUBMISSION)
# Auto-resumes from latest checkpoint if already partially trained.
# ================================================================
import subprocess
print('Training Config D (MAIN): Full loss stack...')
r = subprocess.run(['python', 'train.py', '--config', 'config.yaml', '--ablation', 'full'])
print('Config D', 'DONE' if r.returncode == 0 else 'FAILED')

In [ ]:
# ================================================================
# CELL 7: Evaluate all configs on test split
# ================================================================
import os, subprocess
for ablation in ['l1_only', 'l1_adv', 'l1_adv_fft', 'full']:
    ckpt = f'checkpoints/{ablation}/best.pth'
    if not os.path.exists(ckpt):
        print(f'Skipping {ablation} — no checkpoint')
        continue
    print(f'\n=== Evaluating {ablation} ===')
    subprocess.run(['python', 'eval.py', '--config', 'config.yaml', '--weights', ckpt, '--split', 'test'])
print('\nAll evaluations done!')

In [ ]:
# ================================================================
# CELL 8: Zip and download results
# ================================================================
import shutil, os
shutil.make_archive('/kaggle/working/sar2eo_results',     'zip', '.', 'outputs')
shutil.make_archive('/kaggle/working/sar2eo_checkpoints', 'zip', '.', 'checkpoints')
for f in ['/kaggle/working/sar2eo_results.zip', '/kaggle/working/sar2eo_checkpoints.zip']:
    mb = os.path.getsize(f) / 1e6 if os.path.exists(f) else 0
    print(f'{f}  ({mb:.1f} MB)')
print('Download from the Output panel on the right -->')